In [0]:
##incremental loading

entities = ['customer','driver','location','payments','trips','vehicles']

for entity in entities:
    df_batch = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"/Volumes/pyspark_dbt_cloud/source/source_data/{entity}/")

    schema_entity = df_batch.schema

    df = spark.readStream.format("csv")\
        .option("header", True)\
        .schema(schema_entity)\
        .load(f"/Volumes/pyspark_dbt_cloud/source/source_data/{entity}/")

    df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", f"/Volumes/pyspark_dbt_cloud/bronze/checkpoint/{entity}")\
        .trigger(once=True)\
        .toTable(f"pyspark_dbt_cloud.bronze.{entity}")